# 🏪 Zava Agentic Fine-Tuning Lab — 02: Meet the Agent

**In this notebook**, you'll see Zava's return resolution agent in action.
The agent uses a **tool** (`get_order`) to look up order details, then applies
a complex return policy to determine the correct resolution.

| What you'll do | Time |
|----------------|------|
| Understand the Zava return policy | 2 min |
| Set up the agent (system prompt, tools, runner) | 3 min |
| Run the agent on 3 live scenarios | 5 min |

> **Prerequisite**: Complete `01-introduction-setup.ipynb` first.

---
## Setup — Reconnect to Microsoft Foundry

Each notebook needs its own connection. Run this cell to re-establish the client.

In [ ]:
import json, os, re, time, textwrap
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
print("✅ Connected to Microsoft Foundry")

---
## The Zava Return Policy

This policy is **deliberately complex** — that's what makes it hard for the model
and a great candidate for fine-tuning.

| Rule | Details |
|------|---------|
| Return windows | Standard: 30d, Gold: 45d, Platinum: 60d (electronics: 15d/30d/45d) |
| Electronics restocking | Standard: 15%, Gold: 7.5%, Platinum: 0% |
| Defective items | Always free return, $0 restocking |
| Sale items | Final sale (defective sale → store credit only) |
| Late delivery (>2 days) | $10 shipping credit + 15-day window extension |
| Lost/pending orders | Replacement/refund or cancellation |
| Opened personal care | Deny unless defective |

---
## Define the Agent

The agent consists of four parts:
1. **System prompt** — tells the model what it is and how to apply the policy
2. **Tool definition** — the `get_order` function schema the model can call
3. **Tool executor** — calls the Azure Function (with local fallback)
4. **Agent loop** — orchestrates model → tool call → model → response

In [ ]:
# === Tool endpoint (pre-deployed Azure Function) ===
TOOL_URL = "https://zava-rft-tools.azurewebsites.net"

# The system prompt the agent uses
SYSTEM_PROMPT = """You are Zava's return resolution engine. Call get_order to look up order details, then apply the return policy to compute the resolution.

POLICY: Standard=30d/15d(electronics), Gold=45d/30d, Platinum=60d/45d. Electronics restocking: Std=15%, Gold=7.5%, Plat=0%. Defective=0%. Sale=final sale (defective sale→store credit). Late delivery(>2d)=$10 credit +15d extension. Lost=replacement/refund. Pending=cancellable. Opened personal care=deny unless defective.

Respond with your resolution including: action, amounts, and policy reasoning."""

# Tool definition (same schema the model sees)
TOOLS = [
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up order details including items, prices, dates, loyalty tier, and delivery status.",
        "parameters": {"type": "object", "properties": {
            "order_id": {"type": "string", "description": "The order ID (e.g., ORD-003)"}
        }, "required": ["order_id"]}
    }}
]


def call_tool(name, args):
    """Call the Zava tool endpoint and return the result."""
    url = f"{TOOL_URL}/tool/{name}"
    payload = {"arguments": json.dumps(args), "call_id": "c", "id": "f", "trace_id": "t"}
    r = requests.post(url, json=payload, timeout=30)
    return r.json().get("output", json.dumps(r.json()))


def run_agent(user_message, model="o4-mini", verbose=True):
    """Run the full agent loop: model → tool call → model → response."""
    messages = [
        {"role": "developer", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    tool_calls_made = []

    for turn in range(8):  # max 8 turns to prevent infinite loops
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, max_completion_tokens=8192
        )
        msg = resp.choices[0].message

        # Build assistant message for conversation history
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
            tool_calls_made.extend(msg.tool_calls)
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            if verbose and msg.content:
                print(f"\n📋 Agent Response:\n{textwrap.fill(msg.content, width=80)}")
            return msg.content or "", tool_calls_made

        # Execute tool calls
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  🔧 Calling {tc.function.name}({args})")
            result = call_tool(tc.function.name, args)
            if verbose:
                # Show a preview of the tool result
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  📦 Result: {preview}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "", tool_calls_made


---
## Try it! Run the Agent on Live Scenarios

Now let's see the agent in action. Each scenario sends a customer message, the agent
calls `get_order` to look up the real order data, then reasons about the policy.

**Watch for**:
- Does the agent call the right tool?
- Does it identify the correct action (refund, deny, exchange)?
- Does it compute the right dollar amounts?
- Does it cite the relevant policy rules?

### Scenario 1: Defective Electronics Return (should be straightforward)

In [ ]:
# Scenario 1: Defective headphones
print("=" * 60)
print("SCENARIO 1: Defective headphones")
print("=" * 60)
response, tools = run_agent(
    "Hi, I'm Ava Chen. The headphones from ORD-002 have a cracked speaker. I want a refund."
)

### Scenario 2: Defective Sale Item (tricky policy interaction)

This is where it gets interesting. Sale items are normally final sale, but if they're
**defective**, the customer gets **store credit** (not a refund). The model has to
recognize both the sale status AND the defect to get this right.

In [ ]:
# Scenario 2: Complex — sale item + defective (tricky policy interaction)
print("=" * 60)
print("SCENARIO 2: Defective sale item")
print("=" * 60)
response, tools = run_agent(
    "Emma Kim. The face serum from ORD-004 caused a skin reaction. It was on sale but it's defective."
)

### Scenario 3: Exchange Request (the model often struggles with these)

Exchanges involve computing both the return amount and the new item cost,
plus checking if the return window applies.

In [ ]:
# Scenario 3: Exchange request
print("=" * 60)
print("SCENARIO 3: Exchange request")
print("=" * 60)
response, tools = run_agent(
    "Noah Brown. Exchange hiking boots from ORD-010 for size 11."
)

---
## 💡 Observations

Notice how the agent:
- Calls `get_order` to look up the real order data
- Reasons about the policy to determine the resolution
- Sometimes gets the **action** right but the **amount** wrong
- Sometimes misses a **policy nuance** (e.g., defective sale → store credit, not refund)

These are exactly the kinds of errors that RFT can fix. The model needs to learn
the precise policy rules through trial and error — which is what we'll set up next.

**Next → Open `03-baseline-grader.ipynb` to quantify these errors and understand the grader.**